# Bronze — Yahoo Finance Ingestion

Fetches 2 years of daily market data for ISK exchange rates and the OMX Iceland All-Share Index.

**Source:** Yahoo Finance via `yfinance`  
**Tickers:** `EURUSD=X`, `ISKUSD=X`, `^OMX`  
**Output:** `bronze.yahoo_finance_raw` (Delta table)  
**Schedule:** Daily  

In [ ]:
import yfinance as yf
import pandas as pd
import re

# Create Medallion schemas if they don't exist (idempotent)
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

print("Schemas ready")

In [ ]:
TICKERS = ["EURUSD=X", "ISKUSD=X", "^OMX"]
PERIOD = "2y"
INTERVAL = "1d"

raw_data = yf.download(
    tickers=TICKERS,
    period=PERIOD,
    interval=INTERVAL,
    auto_adjust=True
)

# Flatten MultiIndex columns (price_type, ticker) and sanitise names
raw_data.columns = [
    re.sub(r'[^a-zA-Z0-9]', '_', f"{price}_{ticker}").strip('_').lower()
    for price, ticker in raw_data.columns
]

raw_data = raw_data.reset_index()
raw_data.columns = [
    re.sub(r'[^a-zA-Z0-9]', '_', c).strip('_').lower()
    for c in raw_data.columns
]

raw_data['ingested_at'] = pd.Timestamp.now()

print(f"Fetched {len(raw_data)} rows | Columns: {raw_data.columns.tolist()}")

In [ ]:
spark_df = spark.createDataFrame(raw_data)

spark_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.yahoo_finance_raw")

print(f"Saved to bronze.yahoo_finance_raw — {spark_df.count()} rows")